# 11 · Selektory wycinków: co faktycznie wybierają (MoNuSeg, seed 0)

Wizualna inspekcja wycinków 256×256, które student z danej rundy wybrał do następnej rundy, w trzech przebiegach
AL na wycinkach z kontekstem 64 px (`fullflows_m64`, ten sam `student_init`, te same pierwsze 32 wycinki):

- **random**: kolejne wycinki z permutacji seeda,
- **oracle_peek**: najniższe PQ studenta względem wyroczni na wycinku (czyta etykiety puli, diagnostyka),
- **tta_kd**: średnia rang `K_mean` i `unstable` z D4 TTA (bez etykiet), jak selektor Mateusza na obrazach.

Z 32 wybranych wycinków pokazuję co czwarty (pozycje 0, 4, …, 28 według wyniku selektora, dla random według
permutacji), żeby było widać cały zakres wyników, nie tylko skrajne przypadki.

Każdy wiersz: obraz H&E, kontury **studenta** z rundy wyboru (to, co widział selektor; czerwone), **wyroczni**
(etykiety, które student dostanie; żółte) i **GT MoNuSeg** (zielone). PQ na wycinku: jądra przypisane do wycinka
według środka, dopasowanie na całym obrazie (`patches.tile_quality`).

Dane: `outputs/tile_selection_examples/` (`outputs/export_selection_examples.py`, inferencja na rcmc-gpu).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.segmentation import find_boundaries

from histopathology_pnw.config import Settings

SRC = Settings().outputs_dir / "tile_selection_examples"
METHODS = ["random", "oracle_peek", "tta_kd"]
ROUNDS = sorted({int(p.stem.split("_")[1][1:]) for p in SRC.glob("seed0_r*.csv")})
tables = pd.concat([pd.read_csv(p) for p in sorted(SRC.glob("seed0_r*.csv"))], ignore_index=True)
ROUNDS

## Pokazane wycinki w liczbach

Średnio na 8 pokazanych wycinkach: liczba jąder wyroczni, PQ studenta względem wyroczni (czym kieruje się podgląd)
i PQ wyroczni względem GT (czy etykiety, które student dostaje, są w tym miejscu dobre). 8 wycinków to mała próba:
tabela jest tylko opisem tego, co widać niżej, nie testem.

In [ ]:
summary = (
    tables.groupby(["round", "method"])[["n_oracle", "pq_student_oracle", "pq_oracle_gt"]]
    .mean()
    .unstack("method")
    .reindex(columns=METHODS, level=1)
)
summary.round(3)

In [ ]:
def overlay(ax, img, mask, color):
    ax.imshow(img)
    edge = find_boundaries(mask, mode="thick")
    rgba = np.zeros((*mask.shape, 4))
    rgba[edge] = (*color, 1.0)
    ax.imshow(rgba)
    ax.set_xticks([])
    ax.set_yticks([])


COLORS = {"student": (0.95, 0.15, 0.15), "oracle": (1.0, 0.85, 0.0), "gt": (0.1, 0.9, 0.2)}
TITLES = {
    "image": "H&E",
    "student": "student",
    "oracle": "wyrocznia (etykiety)",
    "gt": "GT MoNuSeg",
}


def show(method, n):
    data = np.load(SRC / f"seed0_r{n:03d}_{method}.npz")
    meta = pd.read_csv(SRC / f"seed0_r{n:03d}_{method}.csv")
    k = len(meta)
    fig, axes = plt.subplots(k, 4, figsize=(11, 2.9 * k))
    for i, row in meta.iterrows():
        img = data["image"][i]
        axes[i, 0].imshow(img)
        axes[i, 0].set_xticks([])
        axes[i, 0].set_yticks([])
        for j, key in enumerate(["student", "oracle", "gt"], start=1):
            overlay(axes[i, j], img, data[key][i], COLORS[key])
        axes[i, 0].set_ylabel(
            f"#{row['pick']} {row['image_id'][:12]} t{row['tile']:02d}\n"
            f"jąder wyr. {row['n_oracle']}, stud. {row['n_student']}\n"
            f"PQ stud/wyr {row['pq_student_oracle']:.2f}\nPQ wyr/GT {row['pq_oracle_gt']:.2f}",
            fontsize=8,
        )
    for j, key in enumerate(TITLES):
        axes[0, j].set_title(TITLES[key], fontsize=10)
    fig.suptitle(f"{method}: wybór studenta z rundy {n} wycinków (do rundy {n + 32})", y=1.0)
    fig.tight_layout()
    plt.show()

## Wybór po rundzie 64 wycinków

Student wytrenowany na 64 wycinkach wybiera 32 wycinki do rundy 96.

### random, runda 64

In [ ]:
show("random", 64)

### oracle_peek, runda 64

In [ ]:
show("oracle_peek", 64)

### tta_kd, runda 64

In [ ]:
show("tta_kd", 64)

## Wybór po rundzie 160 wycinków

Student wytrenowany na 160 wycinkach wybiera 32 wycinki do rundy 192.

### random, runda 160

In [ ]:
show("random", 160)

### oracle_peek, runda 160

In [ ]:
show("oracle_peek", 160)

### tta_kd, runda 160

In [ ]:
show("tta_kd", 160)

## Krzywe na walidacji (seed 0, rundy 32-192)

Ten sam `student_init` i te same pierwsze 32 wycinki, więc w rundzie 32 różnica między przebiegami to sam szum
treningu. Miara główna jak u Mateusza: PQ studenta względem wyroczni na walidacji (`val_oracle_pq`); kontrolnie
PQ względem GT (`val_pq`). Zbiór testowy nie jest używany.

In [ ]:
A = Settings().outputs_dir / "tile_selector_analysis"
res = {m: pd.read_csv(A / f"results_{m}.csv").set_index("n_labeled") for m in METHODS}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["val_oracle_pq", "val_pq"], strict=True):
    for m in METHODS:
        ax.plot(res[m].index, res[m][col], marker="o", label=m)
    ax.set_xlabel("oznaczone wycinki")
    ax.set_title(col)
    ax.grid(alpha=0.3)
axes[0].legend()
plt.show()
curves = pd.concat(
    {
        m: res[m][["val_oracle_pq", "val_pq", "n_labeled_nuclei", "n_source_images"]]
        for m in METHODS
    },
    axis=1,
)
diff = pd.DataFrame(
    {m: res[m]["val_oracle_pq"] - res["random"]["val_oracle_pq"] for m in METHODS[1:]}
)
display(curves.round(4))
display(diff.round(4).rename(columns=lambda c: f"{c} - random"))

## Co wybierały selektory (160 wycinków dobranych po rundzie 32)

Wycinki dodane od rundy 64 do 192 (pierwsze 32 są wspólne), na tle całej puli (480 wycinków):

- `obrazów`, `max na obraz`, `entropia` (bity; równomiernie po 30 obrazach = 4.91): czy wybór się skupia,
- `jąder wyr.`, `jąder GT`: ile etykiet daje wycinek,
- `PQ wyr/GT`: jakość etykiet wyroczni na tych wycinkach,
- `brzeg`: odsetek wycinków stykających się z brzegiem obrazu (w siatce 4x4: 12 z 16 = 0.75).

In [ ]:
pool = pd.read_csv(A / "pool_tiles.csv")
rows = []
for m in METHODS:
    sel = pd.read_csv(A / f"labeled_{m}.csv").merge(pool, on="tile_id").query("added_in > 32")
    c = sel["image_id"].value_counts()
    p = c / c.sum()
    rows.append(
        {
            "selektor": m,
            "obrazów": c.size,
            "max na obraz": c.max(),
            "entropia": -(p * np.log2(p)).sum(),
            "jąder wyr.": sel["n_oracle_owned"].mean(),
            "jąder GT": sel["n_gt"].mean(),
            "PQ wyr/GT": sel["pq_oracle_gt"].mean(),
            "brzeg": sel["edge"].mean(),
            "najczęstsze obrazy": ", ".join(f"{i[:12]}:{n}" for i, n in c.head(4).items()),
        }
    )
rows.append(
    {
        "selektor": "cała pula",
        "obrazów": 30,
        "max na obraz": 16,
        "entropia": np.log2(30),
        "jąder wyr.": pool["n_oracle_owned"].mean(),
        "jąder GT": pool["n_gt"].mean(),
        "PQ wyr/GT": pool["pq_oracle_gt"].mean(),
        "brzeg": pool["edge"].mean(),
    }
)
pd.DataFrame(rows).set_index("selektor").round(3)

## Czy student w ogóle dopasował się do wybranych wycinków

Student z rundy 192 każdego przebiegu segmentuje całą pulę. PQ względem wyroczni osobno na wycinkach, na których był
trenowany, i na pozostałych. Zwykle model wypada lepiej na swoich danych treningowych; jeśli na wycinkach wybranych
przez selektor jest gorzej niż na niewidzianych, to są to miejsca, których nie da się dopasować (etykiety
niejednoznaczne albo niespójne).

In [ ]:
fit = pd.DataFrame(
    {
        m: pd.read_csv(A / f"fit_r192_{m}.csv").groupby("labeled")["pq_student_oracle"].mean()
        for m in METHODS
    }
).rename(index={True: "trenowane (192)", False: "nietrenowane (288)"})
fit.round(3)

## Na co patrzeć

- Czy wycinki podglądu to miejsca, gdzie **wyrocznia** (żółte) wyraźnie różni się od GT (zielone): zlepione
  jądra, pominięte blade jądra, kontury w zrębie. Wtedy podgląd zbiera głównie błędne etykiety (przyczyna 3).
- Czy to wycinki z małą liczbą jąder albo przy brzegu obrazu (wiersz/kolumna 0 lub 3 siatki), gdzie wynik
  jednego wycinka jest zaszumiony (przyczyna 5).
- Czy podgląd i `tta_kd` wybierają podobny typ tkanki (te same obrazy źródłowe), a random coś innego (przyczyny 1, 2).